In [0]:
%run ./utils

# Imports

In [0]:
import pyspark.sql.functions as F

# Load

In [0]:

dbutils.widgets.text("bronze_table_name", "tb_clientes")
bronze_table_name = dbutils.widgets.get("bronze_table_name")

In [0]:
df_clientes = (
    spark.table(f"{bronze_schema}.{bronze_table_name}").
    select(
        "id_cliente",
        "nome",
        "email",
        "regiao",
        "idade",
        "data_cadastro",
        "dia_vencimento_fatura"
    ).distinct()
)

df_clientes = cast_id_to_string(df_clientes)

# Deduplicação

# Tratamento

In [0]:
# "nome": remove espaços nas pontas e padroniza em maiúsculas
df_tratamento = df_clientes.withColumn("nome", F.upper(F.trim(F.col("nome"))))

In [0]:
# "email": emails sem formato válido (ex.: sem @) viram NULL, pois não dá para saber onde estava o @
email_limpo = F.upper(F.trim(F.col("email")))
df_tratamento = df_tratamento.withColumn(
    "email",
    F.when(email_limpo.rlike(r"^[^@\s]+@[^@\s]+\.[^@\s]+$"), email_limpo)
)

In [0]:
# "regiao": remove espaços nas pontas e padroniza em maiúsculas
df_tratamento = df_clientes.withColumn("nome", F.upper(F.trim(F.col("nome"))))

In [0]:
# "idade": conversão para inteiro. Assumindo que cliente precisa ter 18+ e idades acima de 110 são impossíveis
idade_num = F.expr("try_cast(idade as int)")
df_tratamento = df_tratamento.withColumn(
    "idade",
    F.when(idade_num.between(18, 110), idade_num)
)

In [0]:
# "data de cadastro": a origem mistura dois formatos: dd/MM/yyyy e yyyy-MM-dd)
# try_to_date devolve null quando o formato não bate com nenhum dos dois

df_tratamento = df_tratamento.withColumn(
    "data_cadastro",
    F.coalesce(
        F.expr("try_to_date(trim(cast(data_cadastro as string)), 'dd/MM/yyyy')"),
        F.expr("try_to_date(trim(cast(data_cadastro as string)), 'yyyy-MM-dd')")
    )
)

In [0]:
# "dia de vencimento": qualquer dia entre 1 e 28 é válido (todo mês tem pelo menos 28 dias)
# valores fora dessa faixa ou não numéricos viram NULL
venc_num = F.expr("try_cast(dia_vencimento_fatura as int)")
df_limpo = df_tratamento.withColumn(
    "dia_vencimento_fatura",
    F.when(venc_num.between(1, 28), venc_num)
)

# Salvamento

In [0]:
dbutils.widgets.text("silver_table_name", "tb_clientes")
silver_table_name = dbutils.widgets.get("silver_table_name")

In [0]:
save_df(df_limpo, silver_table_name)